# Graphicalizing Aesop's Fables

This notebook downloads the public-domain Project Gutenberg text for [Æsop's Fables #53103](https://www.gutenberg.org/ebooks/53103), splits it into tale-sized documents, and sends those documents through the ontology-guided semantic pipeline.

The notebook uses the package's default OpenAI `gpt-4.1-mini` client. Set `OPENAI_API_KEY` in the environment before running the model cell. The key is read by the OpenAI SDK and is never stored in this notebook.

In [31]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from textwrap import wrap

from IPython.display import display

from semantic_graphicalizer import SemanticGraphicalizer, load_aesop_fables

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent

stories = load_aesop_fables(limit=1, cache_dir=ROOT / 'data' / 'raw', select_at_random=True, rand_seed=None)
[(story.splitlines()[0], len(story)) for story in stories]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[('THE WOLF AND THE LAMB', 817)]

## OpenAI model

The transformer creates the default `OpenAIModelClient` when `model` is omitted. It uses `gpt-4.1-mini` through the Responses API and reads `OPENAI_API_KEY` from the environment.

In [ ]:
from textwrap import wrap

graphicalizer = SemanticGraphicalizer(
    ontology=ROOT / 'configs' / 'ontologies' / 'aesop.yaml',
    prompts=ROOT / 'configs' / 'prompts' / 'aesop.yaml',
)

graphicalizer.fit(stories)
traces = []

for index, story in enumerate(stories, start=1):
    trace = graphicalizer.transform_with_trace([story])[0]
    traces.append(trace)
    title = story.splitlines()[0]
    print(f"\nDocument {index}/{len(stories)}: {title}")
    print("\n".join(wrap(story, width=80)))
    print(f"ID: {trace.document_id}")
    print(f"Entities: {len(trace.entities)} | Relations: {len(trace.relations)} | "
          f"Nodes: {trace.graph.number_of_nodes()} | Edges: {trace.graph.number_of_edges()}")
    print("-" * 80)
    for relation in trace.relations:
        arguments = ", ".join(f"{argument.role}={argument.entity_id}" for argument in relation.arguments)
        print(f"- {relation.type}[{relation.relation}]({arguments})")
    print("\nGraph:")
    display(graphicalizer.display(trace.graph, mode="text"))
    print("=" * 80)


[SemanticGraphicalizer] ready: model=OpenAIModelClient, ontology=aesop-narrative, domain=aesop-narrative
[document-c9dc96ffcafc] segment: 1 -> 1 | 0.0 ms | input_chars=817, chunk_chars=817
[document-c9dc96ffcafc chunk-0] summarize: 1 -> 1 | 2457.1 ms
[document-c9dc96ffcafc chunk-0] normalize: 1 -> 1 | 1898.7 ms
[document-c9dc96ffcafc chunk-0] decompose: 1 -> 14 | 7319.6 ms


In [9]:
graphicalizer.display(
    traces[0].graph,
    mode="dynamic",
    layout="force",
    show_source=True,
    max_width=34,
    width=1200,
    height=760,
    charge_strength=-10,
    link_distance=150,
    component_spacing=100,
    component_strength=0.3,
    parallel_edge_spacing=320
)